[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/int/u2_riemann.ipynb)

# Int. U2 · Sumas de Riemann
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 2 del bloque de cálculo integral. Tiene una sección por subtema, en el mismo orden que el libro:

1. Suma izquierda, derecha y de punto medio
2. Notación sigma y fórmulas cerradas
3. La integral definida como límite de sumas
4. Propiedades básicas de la integral definida
5. Regla del trapecio y regla de Simpson, con tabla de error contra n

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `INT-U2` (bloque, unidad). Las celdas de código de cada sección vienen de `int/u2_riemann/NN_*.py`; los fragmentos que el libro imprime, de `int/u2_riemann/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir funciones usa `x` como variable (en las sumas, `k` es el índice y `n` el límite superior), `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)` y `log(...)` (natural). Los ángulos de `sin`, `cos` y `tan` van en **radianes**.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)
k, n = sp.symbols("k n", integer=True, positive=True)

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
              "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los resultados salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)


def parsear(texto, variables=("x",)):
    """Convierte un texto en una expresión de sympy o explica qué salió mal.
    variables: nombres permitidos (x, k o n)."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una función, por ejemplo: x**2")
    simbolos = {"x": x, "k": k, "n": n}
    local = {**_FUNCIONES, **{v: simbolos[v] for v in variables}}
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa {', '.join(variables)} como variable y, si hace falta, "
                         "sqrt, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la función. Usa * para multiplicar, ** o ^ para potencias "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr) or expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Escribe una expresión sin el signo = y solo con las funciones permitidas.")
    if {str(v) for v in expr.free_symbols} - set(variables):
        raise ValueError(f"Solo puedes usar {', '.join(variables)}.")
    return expr


def lista_numeros(texto, nombre="la lista"):
    """'0, 1.5, 3' -> [0.0, 1.5, 3.0], o un mensaje claro si algo no es número."""
    partes = [p for p in re.split(r"[,\s;]+", str(texto).strip()) if p]
    if not partes:
        raise ValueError(f"Escribe {nombre} como números separados por comas, por ejemplo: 0, 1, 3, 4")
    try:
        return [float(p) for p in partes]
    except ValueError as err:
        raise ValueError(f"En {nombre} hay algo que no es un número: usa punto decimal y comas entre valores.") from err


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def valor_real(f, c):
    """f(c) como float, o ValueError si no es un número real finito."""
    v = complex(sp.N(f.subs(x, c)))
    if abs(v.imag) > 1e-12 or not math.isfinite(v.real):
        raise ValueError(f"f no tiene valor real en x = {cifras(float(c), 6)}: revisa el dominio.")
    return v.real


def numerica(f):
    """Versión numpy de f que siempre devuelve un arreglo del tamaño de la entrada."""
    g = sp.lambdify(x, f, "numpy")
    return lambda t: np.asarray(g(np.asarray(t, dtype=float)), dtype=float) * np.ones_like(np.asarray(t, dtype=float))

## 1. Suma izquierda, derecha y de punto medio

Con $\Delta x=\frac{b-a}{n}$ y $x_k=a+k\,\Delta x$:

$$L_n=\sum_{k=0}^{n-1}f(x_k)\,\Delta x,\qquad R_n=\sum_{k=1}^{n}f(x_k)\,\Delta x,\qquad M_n=\sum_{k=1}^{n}f\!\left(\tfrac{x_{k-1}+x_k}{2}\right)\Delta x.$$

| Si en $[a,b]$… | entonces |
|---|---|
| $f$ crece | $L_n\le A\le R_n$ |
| $f$ decrece | $R_n\le A\le L_n$ |
| $f''>0$ (cóncava hacia arriba) | $M_n\le A$ |
| $f''<0$ (cóncava hacia abajo) | $M_n\ge A$ |

La calculadora revisa el signo de $f'$ y $f''$ en 400 puntos del intervalo y dice qué suma sobreestima. Si el signo cambia, no concluye.

In [ ]:
def sumas_lrm(f, a, b, m):
    """(L_m, R_m, M_m) de la expresión f en [a, b] con m franjas iguales."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    if int(m) != m or m < 1:
        raise ValueError("el número de franjas debe ser un entero positivo.")
    m = int(m)
    for c in np.linspace(a, b, min(m, 50) * 2 + 1):          # detecta puntos fuera del dominio
        valor_real(f, c)
    F = numerica(f)
    xs = np.linspace(a, b, m + 1)
    dx = (b - a) / m
    return float(F(xs[:-1]).sum() * dx), float(F(xs[1:]).sum() * dx), float(F((xs[:-1] + xs[1:]) / 2).sum() * dx)


def _signo_constante(expr, a, b):
    """+1 o -1 si expr no cambia de signo en (a, b); 0 si es idénticamente cero; None si cambia de signo."""
    v = numerica(expr)(np.linspace(a, b, 402)[1:-1])
    if np.allclose(v, 0, atol=1e-12): return 0
    if np.all(v > 0): return 1
    if np.all(v < 0): return -1
    if np.all(v >= 0): return 1      # se anula en puntos aislados (como x^2 en 0)
    if np.all(v <= 0): return -1
    return None


def clasificar_sumas(f, a, b):
    """Texto con qué suma sobreestima y cuál subestima, según monotonía y concavidad."""
    s1, s2 = _signo_constante(sp.diff(f, x), a, b), _signo_constante(sp.diff(f, x, 2), a, b)
    partes = []
    if s1 == 0: partes.append("f es constante: L_n, R_n y M_n son exactas")
    elif s1 == 1: partes.append("f crece: L_n subestima y R_n sobreestima")
    elif s1 == -1: partes.append("f decrece: L_n sobreestima y R_n subestima")
    else: partes.append("f no es monótona en [a, b]: no se puede ordenar L_n y R_n sin más información")
    if s2 == 0 and s1 != 0: partes.append("f'' = 0 (f es una recta): M_n es exacta")
    elif s2 == 1: partes.append("f'' > 0: M_n subestima")
    elif s2 == -1: partes.append("f'' < 0: M_n sobreestima")
    elif s2 is None: partes.append("la concavidad cambia: no se puede decir si M_n sobreestima o subestima")
    return "; ".join(partes)


def calculadora_sumas(f_txt, a, b, m, regla, n_cifras):
    try:
        f = parsear(f_txt)
        L, R, M = sumas_lrm(f, a, b, m)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err); return
    print(f"L_{m} ≈ {cifras(L, n_cifras)}   R_{m} ≈ {cifras(R, n_cifras)}   M_{m} ≈ {cifras(M, n_cifras)}"
          f"   ({n_cifras} cifras significativas, redondeado)")
    print(clasificar_sumas(f, a, b))
    F = numerica(f)
    xs = np.linspace(a, b, m + 1); dx = (b - a) / m
    pos = {"izquierda": xs[:-1], "derecha": xs[1:], "punto medio": (xs[:-1] + xs[1:]) / 2}[regla]
    fig, ax = plt.subplots(figsize=(5.5, 3))
    for x0, xm in zip(xs[:-1], pos):
        ax.add_patch(plt.Rectangle((x0, 0), dx, F(np.array([xm]))[0], facecolor="0.85", edgecolor="black"))
    ax.plot(pos, F(pos), "o", color="red")
    xx = np.linspace(a, b, 300); ax.plot(xx, F(xx), color="navy"); ax.axhline(0, color="black", lw=0.7)
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    ax.set_title(f"suma {regla}, n = {m}"); plt.show()


widgets.interact(calculadora_sumas,
    f_txt=widgets.Text(value="x**2", description="f(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=1, description="b"),
    m=widgets.IntSlider(value=4, min=1, max=60, description="n"),
    regla=widgets.Dropdown(options=["izquierda", "derecha", "punto medio"], value="punto medio", description="regla"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
_L, _R, _M = sumas_lrm(x**2, 0, 1, 4)
PRUEBAS_1 = [
    ("x^2 en [0, 1], n = 4: L = 0.21875, R = 0.46875, M = 0.328125",
     lambda: cerca(_L, 0.21875) and cerca(_R, 0.46875) and cerca(_M, 0.328125)),
    ("1/x en [1, 2], n = 4: M ≈ 0.691220", lambda: cerca(sumas_lrm(1 / x, 1, 2, 4)[2], 0.6912198912, 1e-9)),
    ("constante 2 en [0, 5]: las tres dan 10", lambda: all(cerca(v, 10) for v in sumas_lrm(2 + 0 * x, 0, 5, 3))),
    ("recta 5x + 2 en [0, 2], n = 3: M exacto = 14", lambda: cerca(sumas_lrm(5 * x + 2, 0, 2, 3)[2], 14)),
    ("hidrograma 30/(1 + x): decrece y f'' > 0", lambda: clasificar_sumas(30 / (1 + x), 0, 4).startswith("f decrece")
                                                     and "M_n subestima" in clasificar_sumas(30 / (1 + x), 0, 4)),
    ("recta 4x - 3: f'' = 0 y M_n exacta", lambda: "M_n es exacta" in clasificar_sumas(4 * x - 3, 1, 2)),
    ("a >= b, n = 0 o f fuera del dominio se rechazan", lambda: _rechaza(lambda: sumas_lrm(x, 1, 0, 4))
                                                              and _rechaza(lambda: sumas_lrm(x, 0, 1, 0))
                                                              and _rechaza(lambda: sumas_lrm(sp.sqrt(x), -1, 1, 4))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano $L_3$, $R_3$ y $M_3$ para $f(x)=2x+1$ en $[0,3]$.

In [ ]:
mi_L3 = None    # escribe un número
mi_R3 = None    # escribe un número
mi_M3 = None    # escribe un número

if None in (mi_L3, mi_R3, mi_M3):       # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = sumas_lrm(2 * x + 1, 0, 3, 3)
    print("La calculadora da: L_3 =", cifras(ref[0], 4), " R_3 =", cifras(ref[1], 4), " M_3 =", cifras(ref[2], 4))
    print("coinciden" if all(cerca(u, v, 1e-6) for u, v in zip((mi_L3, mi_R3, mi_M3), ref)) else
          "NO coinciden: con Δx = 1, las alturas izquierdas son f(0), f(1), f(2) y las de punto medio f(0.5), f(1.5), f(2.5)")

## 2. Notación sigma y fórmulas cerradas

$$\sum_{k=1}^{n}k=\frac{n(n+1)}{2},\qquad \sum_{k=1}^{n}k^2=\frac{n(n+1)(2n+1)}{6},\qquad \sum_{k=1}^{n}k^3=\Big[\frac{n(n+1)}{2}\Big]^2,\qquad \sum_{k=1}^{n}c=n\,c.$$

La calculadora recibe el término general en `k` y los límites. Si el límite superior es `n`, devuelve la fórmula cerrada y la **comprueba** con la suma término a término para $n=1,2,3$: una fórmula que falla con $n=1$ está mal, por elegante que se vea.

In [ ]:
def suma_sigma(termino_txt, inferior, superior_txt):
    """Valor exacto de la suma (si el límite superior es un entero) o fórmula cerrada en n (si es 'n')."""
    t = parsear(termino_txt, variables=("k", "n"))
    if int(inferior) != inferior:
        raise ValueError("el límite inferior debe ser un entero.")
    inferior = int(inferior)
    sup = str(superior_txt).strip()
    if sup == "n":
        cerrada = sp.factor(sp.simplify(sp.summation(t, (k, inferior, n))))
        for m in range(max(1, inferior), max(1, inferior) + 3):        # comprobación término a término
            directa = sum(t.subs({k: j, n: m}) for j in range(inferior, m + 1))
            if sp.simplify(cerrada.subs(n, m) - directa) != 0:
                raise ValueError(f"la fórmula no coincide con la suma término a término en n = {m}.")
        return cerrada
    try:
        s_ = int(sup)
    except ValueError as err:
        raise ValueError("el límite superior debe ser un entero o la letra n.") from err
    if s_ < inferior:
        raise ValueError("el límite superior debe ser mayor o igual que el inferior.")
    if t.free_symbols - {k}:
        raise ValueError("con un límite superior numérico, el término solo puede depender de k.")
    return sp.nsimplify(sum(t.subs(k, j) for j in range(inferior, s_ + 1)))


def calculadora_sigma(termino_txt, inferior, superior_txt):
    try:
        r = suma_sigma(termino_txt, inferior, superior_txt)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err); return
    if str(superior_txt).strip() == "n":
        print(f"fórmula cerrada: {r}   (comprobada término a término para tres valores de n)")
    else:
        sup = int(str(superior_txt).strip())
        print(f"suma de {sup - int(inferior) + 1} términos = {r}")


widgets.interact(calculadora_sigma,
    termino_txt=widgets.Text(value="k**2", description="término"),
    inferior=widgets.IntText(value=1, description="desde k ="),
    superior_txt=widgets.Text(value="n", description="hasta"));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("suma de k de 1 a 100 = 5050", lambda: suma_sigma("k", 1, "100") == 5050),
    ("suma de k^3 de 1 a n = (n(n+1)/2)^2", lambda: sp.simplify(suma_sigma("k**3", 1, "n") - (n * (n + 1) / 2)**2) == 0),
    ("suma de 3 de 1 a n = 3n (no 3)", lambda: suma_sigma("3", 1, "n") == 3 * n),
    ("suma de 3k^2 - 2k + 4 = n^3 + n^2/2 + 7n/2", lambda: sp.simplify(suma_sigma("3*k**2 - 2*k + 4", 1, "n")
                                                                       - (n**3 + n**2 / 2 + 7 * n / 2)) == 0),
    ("de k = 4 a 9 hay 6 términos; superior menor que inferior se rechaza",
     lambda: suma_sigma("1", 4, "9") == 6 and _rechaza(lambda: suma_sigma("k", 5, "2"))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano $\displaystyle\sum_{k=1}^{6}\big(k^2-k\big)$ con las fórmulas cerradas.

In [ ]:
mi_suma = None      # escribe un número

if mi_suma is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = suma_sigma("k**2 - k", 1, "6")
    print("La calculadora da:", ref)
    print("coinciden" if mi_suma == ref else "NO coinciden: separa la suma en suma de k^2 menos suma de k, con n = 6")

## 3. La integral definida como límite de sumas

$$\int_a^b f(x)\,dx=\lim_{n\to\infty}\sum_{k=1}^{n}f(x_k^*)\,\Delta x,\qquad \Delta x=\frac{b-a}{n}.$$

Para un polinomio, la suma derecha $R_n$ tiene fórmula cerrada en $n$ (con las fórmulas de $\sum k^p$) y el límite sale con álgebra. La calculadora escribe $R_n$, la simplifica, toma el límite y muestra $R_{10}$, $R_{100}$ y $R_{1000}$ para ver la convergencia en números.

In [ ]:
def suma_derecha_simbolica(f, a, b):
    """R_n en forma cerrada (en n) y su límite, para f polinomial o simple."""
    a, b = sp.nsimplify(a), sp.nsimplify(b)
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    dx = (b - a) / n
    R = sp.simplify(sp.summation(f.subs(x, a + k * dx) * dx, (k, 1, n)))
    if R.has(sp.Sum):
        raise ValueError("sympy no encontró una fórmula cerrada para R_n; prueba con un polinomio.")
    return sp.factor(R), sp.limit(R, n, sp.oo)


def calculadora_limite(f_txt, a, b, n_cifras):
    try:
        f = parsear(f_txt)
        R, lim = suma_derecha_simbolica(f, a, b)
    except (ValueError, TypeError, NotImplementedError) as err:
        print("Revisa la entrada:", err); return
    an, bn = sp.nsimplify(a), sp.nsimplify(b)
    print("R_n =", sp.Sum(f.subs(x, an + k * (bn - an) / n) * (bn - an) / n, (k, 1, n)))
    print("    =", R)
    for m in (10, 100, 1000):
        print(f"  R_{m} ≈ {cifras(R.subs(n, m), n_cifras)}")
    print(f"límite cuando n → ∞: {lim} ≈ {cifras(lim, n_cifras)}   ({n_cifras} cifras significativas, redondeado)")


widgets.interact(calculadora_limite,
    f_txt=widgets.Text(value="x**2", description="f(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=1, description="b"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=10, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("x^2 en [0, 1]: R_n = (n+1)(2n+1)/(6n^2) y límite 1/3",
     lambda: sp.simplify(suma_derecha_simbolica(x**2, 0, 1)[0] - (n + 1) * (2 * n + 1) / (6 * n**2)) == 0
             and suma_derecha_simbolica(x**2, 0, 1)[1] == sp.Rational(1, 3)),
    ("x en [0, 3]: límite 9/2 (triángulo)", lambda: suma_derecha_simbolica(x, 0, 3)[1] == sp.Rational(9, 2)),
    ("x^3 en [0, 2]: límite 4", lambda: suma_derecha_simbolica(x**3, 0, 2)[1] == 4),
    ("constante 5 en [1, 4]: 15", lambda: suma_derecha_simbolica(5 + 0 * x, 1, 4)[1] == 15),
    ("a >= b se rechaza", lambda: _rechaza(lambda: suma_derecha_simbolica(x, 2, 2))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Escribe a mano $R_n$ para $f(x)=2x$ en $[0,1]$, simplifícala con la fórmula de $\sum k$ y toma el límite.

In [ ]:
mi_Rn = None       # escribe la expresión en n como texto, por ejemplo: "3 - 1/n"
mi_limite = None   # escribe un número

if mi_Rn is None or mi_limite is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    R, lim = suma_derecha_simbolica(2 * x, 0, 1)
    print("La calculadora da: R_n =", R, "  límite =", lim)
    try:
        ok = sp.simplify(parsear(str(mi_Rn), variables=("n",)) - R) == 0 and cerca(mi_limite, lim, 1e-9)
    except ValueError as err:
        print(err); ok = False
    print("coinciden" if ok else "NO coinciden: R_n = suma de 2(k/n)(1/n) = (2/n^2)·n(n+1)/2")

## 4. Propiedades básicas de la integral definida

* Convenios: $\int_a^a f=0$ y $\int_b^a f=-\int_a^b f$.
* Linealidad: $\int(\alpha f+\beta g)=\alpha\int f+\beta\int g$. Aditividad: $\int_a^b=\int_a^c+\int_c^b$ (aunque $c$ quede fuera).
* Cotas: si $m\le f\le M$ en $[a,b]$, entonces $m(b-a)\le\int_a^b f\le M(b-a)$.
* Simetría en $[-a,a]$: par, $2\int_0^a f$; impar, $0$.

La calculadora comprueba cada propiedad con `sympy` (exacto) y con una suma de punto medio (numérico). Que coincidan las dos vías es la comprobación.

In [ ]:
def integral_exacta(f, a, b):
    return sp.integrate(f, (x, sp.nsimplify(a), sp.nsimplify(b)))


def integral_numerica(f, a, b, m=20000):
    """Suma de punto medio con m franjas; acepta a > b (Δx negativo)."""
    F = numerica(f)
    xs = np.linspace(a, b, m + 1)
    return float(F((xs[:-1] + xs[1:]) / 2).sum() * (b - a) / m)


def extremos(f, a, b):
    """(m, M) de f en [a, b] con los extremos del intervalo y los puntos donde f' = 0."""
    cand = [sp.nsimplify(a), sp.nsimplify(b)]
    crit = sp.solveset(sp.diff(f, x), x, domain=sp.Interval.open(sp.nsimplify(a), sp.nsimplify(b)))
    if isinstance(crit, sp.FiniteSet):
        cand += list(crit)
    elif crit is not sp.S.EmptySet:
        cand += list(np.linspace(float(a), float(b), 401))
    vals = [valor_real(f, c) for c in cand]
    return min(vals), max(vals)


def revisar_propiedades(f, g, a, b, c, alfa, beta):
    """Diccionario {propiedad: (lado izquierdo, lado derecho)} evaluado con sympy."""
    I = lambda h, p, q: integral_exacta(h, p, q)
    r = {"linealidad": (I(alfa * f + beta * g, a, b), alfa * I(f, a, b) + beta * I(g, a, b)),
         "aditividad": (I(f, a, b), I(f, a, c) + I(f, c, b)),
         "límites al revés": (I(f, b, a), -I(f, a, b))}
    if a < b:
        m_, M_ = extremos(f, a, b)
        r["cotas"] = (m_ * (b - a), float(I(f, a, b)), M_ * (b - a))
    return r


def calculadora_propiedades(f_txt, g_txt, a, b, c, alfa, beta, n_cifras):
    try:
        f, g = parsear(f_txt), parsear(g_txt)
        r = revisar_propiedades(f, g, a, b, c, alfa, beta)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err); return
    for nombre, valores in r.items():
        if nombre == "cotas":
            print(f"  cotas: {cifras(valores[0], n_cifras)} <= {cifras(valores[1], n_cifras)} <= {cifras(valores[2], n_cifras)}")
        else:
            izq, der = valores
            ok = "se cumple" if sp.simplify(izq - der) == 0 else "NO se cumple"
            print(f"  {nombre}: {cifras(izq, n_cifras)} y {cifras(der, n_cifras)} -> {ok}")
    print(f"  comprobación numérica de ∫ f de a a b (punto medio): {cifras(integral_numerica(f, a, b), n_cifras)}")
    print(f"  ({n_cifras} cifras significativas, redondeado)")


widgets.interact(calculadora_propiedades,
    f_txt=widgets.Text(value="x**2", description="f(x) ="), g_txt=widgets.Text(value="x", description="g(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=2, description="b"),
    c=widgets.FloatText(value=3, description="c"),
    alfa=widgets.FloatText(value=3, description="α"), beta=widgets.FloatText(value=-2, description="β"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("linealidad: ∫(3x^2 - 2x + 5) en [0, 1] = 5", lambda: integral_exacta(3 * x**2 - 2 * x + 5, 0, 1) == 5),
    ("aditividad con c fuera: ∫_1^2 x^2 = ∫_1^3 + ∫_3^2 = 7/3",
     lambda: integral_exacta(x**2, 1, 3) + integral_exacta(x**2, 3, 2) == sp.Rational(7, 3)),
    ("límites al revés: ∫_2^1 x^2 = -7/3, también con sumas",
     lambda: integral_exacta(x**2, 2, 1) == -sp.Rational(7, 3) and cerca(integral_numerica(x**2, 2, 1), -7 / 3, 1e-6)),
    ("cotas de e^(-x^2) en [0, 1]: e^-1 y 1", lambda: cerca(extremos(sp.exp(-x**2), 0, 1)[0], math.exp(-1))
                                                   and cerca(extremos(sp.exp(-x**2), 0, 1)[1], 1)),
    ("simetría: ∫x^3 en [-2, 2] = 0 y ∫x^2 en [-2, 2] = 16/3",
     lambda: integral_exacta(x**3, -2, 2) == 0 and integral_exacta(x**2, -2, 2) == sp.Rational(16, 3)),
    ("la cota de x^2 + 1 en [0, 1] descarta 1/3", lambda: revisar_propiedades(x**2 + 1, x, 0, 1, 2, 1, 1)["cotas"][0] > 1 / 3),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano $\int_{-1}^{1}\big(x^3+2\big)dx$ con la simetría y la integral de una constante.

In [ ]:
mi_integral = None     # escribe un número

if mi_integral is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = integral_exacta(x**3 + 2, -1, 1)
    print("La calculadora da:", ref)
    print("coinciden" if cerca(mi_integral, ref, 1e-9) else "NO coinciden: x^3 es impar y la constante 2 aporta 2·(1 - (-1))")

## 5. Regla del trapecio y regla de Simpson, con tabla de error contra $n$

$$T_n=\frac{\Delta x}{2}\big[y_0+2y_1+\cdots+2y_{n-1}+y_n\big],\qquad S_n=\frac{\Delta x}{3}\big[y_0+4y_1+2y_2+\cdots+4y_{n-1}+y_n\big]\ \ (n\ \text{par}).$$

Cotas: $|E_T|\le\dfrac{K_2(b-a)^3}{12n^2}$ y $|E_S|\le\dfrac{K_4(b-a)^5}{180n^4}$. Al duplicar $n$, el error del trapecio se divide entre $\approx4$ y el de Simpson entre $\approx16$.

La calculadora acepta una **función** o una **tabla** de lecturas igualmente espaciadas. Con una tabla de $N$ lecturas hay $N-1$ franjas; Simpson exige que ese número sea par.

In [ ]:
def trapecio_datos(y, h):
    y = np.asarray(y, dtype=float)
    if not h > 0:
        raise ValueError("el ancho h de cada franja debe ser positivo.")
    if y.size < 2:
        raise ValueError("se necesitan al menos dos lecturas.")
    return float(h * (y.sum() - (y[0] + y[-1]) / 2))


def simpson_datos(y, h):
    y = np.asarray(y, dtype=float)
    if not h > 0:
        raise ValueError("el ancho h de cada franja debe ser positivo.")
    franjas = y.size - 1
    if franjas < 2 or franjas % 2:
        raise ValueError(f"hay {franjas} franjas: Simpson necesita un número par (un número impar de lecturas). "
                         "Usa el trapecio o combina Simpson con un trapecio en la última franja.")
    return float(h / 3 * (y[0] + y[-1] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum()))


def reglas_funcion(f, a, b, m):
    """(T_m, S_m) de la expresión f; S_m es None si m es impar."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    if int(m) != m or m < 1:
        raise ValueError("el número de franjas debe ser un entero positivo.")
    m = int(m)
    xs = np.linspace(a, b, m + 1)
    y = [valor_real(f, c) for c in xs]
    return trapecio_datos(y, (b - a) / m), (simpson_datos(y, (b - a) / m) if m % 2 == 0 else None)


def cota_maxima(expr, a, b):
    """Máximo de |expr| en [a, b], muestreado en 2001 puntos (para K_2 y K_4); inf si no está acotada."""
    with np.errstate(all="ignore"):
        v = np.abs(numerica(expr)(np.linspace(a, b, 2001)))
    return float(np.max(v)) if np.all(np.isfinite(v)) else math.inf


def texto_cota(K, potencia, divisor, a, b, m, n_cifras):
    if not math.isfinite(K):
        return "no hay cota: la derivada que pide la fórmula no está acotada en [a, b]"
    return cifras(K * (b - a)**potencia / (divisor * m**(potencia - 1)), n_cifras)


def tabla_error(f, a, b, exacto, ns=(2, 4, 8, 16, 32)):
    """Filas (n, T_n, error T, razón T, S_n, error S, razón S)."""
    filas, prev = [], None
    for m in ns:
        T, S = reglas_funcion(f, a, b, m)
        eT, eS = T - float(exacto), S - float(exacto)
        rT = abs(prev[0] / eT) if prev and eT else None
        rS = abs(prev[1] / eS) if prev and eS else None
        filas.append((m, T, eT, rT, S, eS, rS)); prev = (eT, eS)
    return filas


def calculadora_trapecio_simpson(modo, f_txt, a, b, m, datos_txt, h, n_cifras):
    try:
        if modo == "función":
            f = parsear(f_txt)
            T, S = reglas_funcion(f, a, b, m)
            K2, K4 = cota_maxima(sp.diff(f, x, 2), a, b), cota_maxima(sp.diff(f, x, 4), a, b)
            print(f"({n_cifras} cifras significativas, redondeado; la tabla de error usa 8)")
            print(f"T_{m} ≈ {cifras(T, n_cifras)}   cota del error: {texto_cota(K2, 3, 12, a, b, m, n_cifras)}")
            if S is None:
                print(f"S_{m}: n impar, Simpson no aplica")
            else:
                print(f"S_{m} ≈ {cifras(S, n_cifras)}   cota del error: {texto_cota(K4, 5, 180, a, b, m, n_cifras)}")
            exacto = integral_exacta(f, a, b)
            if exacto.is_number and not exacto.has(sp.Integral):
                print(f"valor exacto (sympy): {exacto} ≈ {cifras(exacto, n_cifras)}")
                print("   n          T_n      error T   razón        S_n      error S   razón")
                for fila in tabla_error(f, a, b, exacto):
                    mm, T_, eT, rT, S_, eS, rS = fila
                    print(f"{mm:4d}  {cifras(T_, 8):>11}  {eT:10.2e}  {('%.2f' % rT) if rT else '':>6}"
                          f"  {cifras(S_, 8):>11}  {eS:10.2e}  {('%.2f' % rS) if rS else '':>6}")
        else:
            y = lista_numeros(datos_txt, "las lecturas")
            T = trapecio_datos(y, h)            # valida h y el número de lecturas antes de imprimir
            print(f"{len(y)} lecturas, {len(y) - 1} franjas de ancho {h}   ({n_cifras} cifras significativas, redondeado)")
            print(f"trapecio ≈ {cifras(T, n_cifras)}")
            print(f"Simpson ≈ {cifras(simpson_datos(y, h), n_cifras)}")
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err)


widgets.interact(calculadora_trapecio_simpson,
    modo=widgets.Dropdown(options=["función", "tabla"], value="función", description="modo"),
    f_txt=widgets.Text(value="1/x", description="f(x) ="),
    a=widgets.FloatText(value=1, description="a"), b=widgets.FloatText(value=2, description="b"),
    m=widgets.IntSlider(value=4, min=1, max=64, description="n"),
    datos_txt=widgets.Text(value="40, 42, 45, 44, 41", description="lecturas"),
    h=widgets.FloatText(value=2, description="ancho h"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=10, description="cifras sig."));

**Las cuatro reglas lado a lado.** La tabla de la figura 2.8 del libro: error de $L_n$, $M_n$, $T_n$ y $S_n$ para $P(0\le Z\le1)=\int_0^1\varphi(x)\,dx$ y la razón entre errores consecutivos. Al duplicar $n$, las razones se acercan a 2, 4, 4 y 16.

In [ ]:
def tabla_cuatro_reglas(f, a, b, exacto, ns=(2, 4, 8, 16, 32, 64)):
    F, ex = numerica(f), float(exacto)
    filas, previo = [], None
    for m in ns:
        xs = np.linspace(a, b, m + 1); h = (b - a) / m; y = F(xs)
        valores = (h * y[:-1].sum(), h * F((xs[:-1] + xs[1:]) / 2).sum(), trapecio_datos(y, h), simpson_datos(y, h))
        errores = [v - ex for v in valores]
        razones = [abs(p / e) if previo else None for p, e in zip(previo or errores, errores)]
        filas.append((m, errores, razones)); previo = errores
    return filas


_phi = sp.exp(-x**2 / 2) / sp.sqrt(2 * sp.pi)
_tabla4 = tabla_cuatro_reglas(_phi, 0, 1, sp.erf(1 / sp.sqrt(2)) / 2)
print("   n      error L      error M      error T      error S   razones L, M, T, S")
for m, errores, razones in _tabla4:
    r = "" if razones[0] is None else "  ".join(f"{q:5.2f}" for q in razones)
    print(f"{m:4d}  " + "  ".join(f"{e:11.3e}" for e in errores) + "   " + r)

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("x^2 en [0, 1]: S_2 = 1/3 exacto (Simpson es exacto en cúbicas)", lambda: cerca(reglas_funcion(x**2, 0, 1, 2)[1], 1 / 3)),
    ("1/x en [1, 2], n = 4: T ≈ 0.697024, S ≈ 0.693254",
     lambda: cerca(reglas_funcion(1 / x, 1, 2, 4)[0], 0.6970238095, 1e-9) and cerca(reglas_funcion(1 / x, 1, 2, 4)[1], 0.6932539683, 1e-9)),
    ("e^x en [0, 1], n = 2: T ≈ 1.75393, S ≈ 1.71886",
     lambda: cerca(reglas_funcion(sp.exp(x), 0, 1, 2)[0], 1.7539310925, 1e-9) and cerca(reglas_funcion(sp.exp(x), 0, 1, 2)[1], 1.7188611519, 1e-9)),
    ("tabla constante de 6 lecturas: trapecio 25 y Simpson rechazado",
     lambda: cerca(trapecio_datos([5] * 6, 1), 25) and _rechaza(lambda: simpson_datos([5] * 6, 1))),
    ("tabla de las cuatro reglas para la normal: razones ≈ 2, 4, 4 y 16 con n = 64",
     lambda: all(abs(q - r) < t for q, r, t in zip(_tabla4[-1][2], (2, 4, 4, 16), (0.05, 0.05, 0.05, 0.3)))),
    ("razones de la tabla de error para 1/x: ≈ 4 (trapecio) y ≈ 16 (Simpson)",
     lambda: abs(tabla_error(1 / x, 1, 2, sp.log(2))[-1][3] - 4) < 0.05 and abs(tabla_error(1 / x, 1, 2, sp.log(2))[-1][6] - 16) < 0.5),
    ("la cota del trapecio para 1/x con n = 409 es menor que 1e-6",
     lambda: cota_maxima(sp.diff(1 / x, x, 2), 1, 2) / (12 * 409**2) < 1e-6),
    ("a >= b o h <= 0 se rechazan; √x en [0, 1] no tiene cota", lambda: _rechaza(lambda: reglas_funcion(x, 1, 1, 4))
                                                                 and _rechaza(lambda: trapecio_datos([1, 2], 0))
                                                                 and math.isinf(cota_maxima(sp.diff(sp.sqrt(x), x, 2), 0, 1))),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Retoma tu predicción de la semilla.** La celda siguiente cuenta cuántas franjas necesita cada regla para que $\int_0^1\frac{4}{1+x^2}dx$ dé $\pi$ con seis cifras ($|E|<5\times10^{-6}$). La suma izquierda tarda unos segundos: son muchos rectángulos.

In [ ]:
def _reglas_pi(m, regla):
    xs = np.linspace(0, 1, m + 1); y = 4 / (1 + xs**2); h = 1 / m
    if regla == "L": return h * y[:-1].sum()
    if regla == "M": xm = (xs[:-1] + xs[1:]) / 2; return h * (4 / (1 + xm**2)).sum()
    if regla == "T": return trapecio_datos(y, h)
    return simpson_datos(y, h)


def minimo_n(regla, tol=5e-6):
    paso = 2 if regla == "S" else 1
    lo = paso
    while abs(_reglas_pi(lo, regla) - math.pi) >= tol:     # duplica hasta pasarse
        lo *= 2
    hi, lo = lo, max(paso, lo // 2)
    while hi - lo > paso:                                  # bisección sobre n
        mid = (lo + hi) // 2
        mid -= mid % paso
        if abs(_reglas_pi(mid, regla) - math.pi) < tol: hi = mid
        else: lo = mid
    return hi


for regla, nombre in (("L", "suma izquierda"), ("M", "punto medio"), ("T", "trapecio"), ("S", "Simpson")):
    print(f"{nombre:15s}: n = {minimo_n(regla)}")

**Contrasta (sección 5).** Calcula a mano $T_2$ y $S_2$ para $\int_0^2x^3\,dx$ y compáralos con el valor exacto, $4$.

In [ ]:
mi_T2 = None    # escribe un número
mi_S2 = None    # escribe un número

if mi_T2 is None or mi_S2 is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    T2, S2 = reglas_funcion(x**3, 0, 2, 2)
    print("La calculadora da: T_2 =", cifras(T2, 6), "  S_2 =", cifras(S2, 6))
    print("coinciden" if cerca(mi_T2, T2, 1e-9) and cerca(mi_S2, S2, 1e-9) else
          "NO coinciden: con Δx = 1, las lecturas son f(0) = 0, f(1) = 1 y f(2) = 8")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")